# Week 22 — Constrained MDPs, PPO-Lagrangian, CPO/FOCOPS/P3O

This workbook has two explicit exercise modes, controlled by `REFERENCE_MODE`
in the setup cell.

- **Reference (`True`, default):** supplied implementations demonstrate the
  calculations. Passing these checks verifies the reference, not your code.
- **Learner (`False`):** complete each scaffold's function, then restart and
  run all cells. The corresponding checks call your function. An unfinished
  scaffold raises `NotImplementedError`; there is no reference fallback.
  In a reference-wrapper scaffold, replace its guarded placeholder body with
  your implementation before choosing learner mode.

Change the setup flag before restarting. Record a boundary-case prediction
before each check and change an input after it passes. Later experiment cells
without a scaffold are reference demonstrations, not automated learner grades.

**Teaching companion:** [Worked examples, assumptions and self-checks](teaching_guide.md).
For additional independently graded tasks, see the
[progressive coding projects](../docs/INTERACTIVE_LEARNING.md).


**A bounded first investigation:** Draw the feasible reward–cost tradeoff. Start with **Exercises 1, 3 and 5** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-draw-the-feasible-rewardcost-tradeoff).
Save your own prediction before running the reference cells. For your own implementation, choose learner mode in setup, restart, and verify
that the corresponding check calls your completed function.


**Worked examples and derivations:** [Teaching guide](teaching_guide.md) — prerequisite links, analogy boundaries, calculated examples, code connections, and retrieval questions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, matplotlib.pyplot as plt
from project import solutions as ref
# True runs documented references. False checks YOUR completed functions.
# Restart and run all after changing this flag; an unfinished learner stops.
REFERENCE_MODE = True
def select_exercise(reference, learner):
    return reference if REFERENCE_MODE else learner
print("Exercise mode:", "reference demonstration" if REFERENCE_MODE else "learner checks (no fallback)")


## Exercise 1 ★ — Lagrangian dual step

Derive on paper the projected-gradient-descent update on $\lambda$ for $L(\pi,\lambda) = J_r - \lambda(J_c - d)$.

> **Hint** (try yourself first): descent on this Lagrangian gives $\lambda \leftarrow [\lambda + \eta(J_c - d)]_+$. Sign matters.

In [ ]:
def exercise_1(lam, eta, J_c, d):
    # YOUR CODE HERE — replace with your derivation
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_1 before running learner checks")
    return ref.exercise_1(lam, eta, J_c, d)


In [ ]:
assert exercise_1(0.5, 0.1, 2.0, 1.0) > 0.5
assert exercise_1(0.05, 1.0, 0.0, 1.0) == 0.0
print('ok')

## Exercise 2 ★ — P3O hinge subgradient

Return a subgradient of $\kappa\cdot[J_c - d]_+$ w.r.t.\ $J_c$.

> **Hint**: it is $\kappa$ above $d$, $0$ below $d$, anything in $[0,\kappa]$ at the kink.

In [ ]:
def exercise_2(J_c, d, kappa):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_2 before running learner checks")
    return ref.exercise_2(J_c, d, kappa)


In [ ]:
assert exercise_2(2.0, 1.0, 10.0) == 10.0
assert exercise_2(0.5, 1.0, 10.0) == 0.0
print('ok')

## Exercise 3 ★★ — GAE on the cost stream

Implement Generalised Advantage Estimation (Schulman et al. 2016). PPO-Lagrangian calls it twice: once on reward, once on cost.

> **Hint**: iterate from the end; $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ and $A_t = \delta_t + \gamma\lambda A_{t+1}$ when nonterminal.

In [ ]:
def exercise_3(rews, vals, dones, gamma, lam, last_val=0.0):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_3 before running learner checks")
    return ref.exercise_3(rews, vals, dones, gamma, lam, last_val)


In [ ]:
rews=[0.1]*10; vals=[0.0]*10; dones=[False]*9+[True]
adv, ret = exercise_3(rews, vals, dones, 0.9, 0.95)
assert adv.shape == (10,)
print('ok')
fixture_adv,fixture_ret=exercise_3([1.,2.],[0.,0.],[False,True],.9,1.)
np.testing.assert_allclose(fixture_adv,[2.8,2.])
np.testing.assert_allclose(fixture_ret,[2.8,2.])


## Exercise 4 ★★ — FOCOPS closed-form non-parametric update

Derive and implement $\pi^\star(a|s)\propto \pi_{\rm old}(a|s)\exp((A_r - \nu A_c)/\lambda)$.

> **Hint**: log-space + softmax for numerical stability.

In [ ]:
def exercise_4(pi_old, A_r, A_c, lam, nu):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_4 before running learner checks")
    return ref.exercise_4(pi_old, A_r, A_c, lam, nu)


In [ ]:
pi = np.full((4,3), 1/3.0)
A_r = np.random.RandomState(0).randn(4,3); A_c = np.random.RandomState(1).randn(4,3)
new = exercise_4(pi, A_r, A_c, 1.0, 0.5)
assert np.allclose(new.sum(1), 1.0); assert (new >= 0).all()
print('ok')
fixture=exercise_4(np.array([[.5,.5]]),np.array([[0.,np.log(2)]]),np.zeros((1,2)),1.,0.)
np.testing.assert_allclose(fixture,[[1/3,2/3]])


## Exercise 5 ★★ — Empirical CVaR

Approximate lower-tail $\mathrm{CVaR}_\alpha(G)$ from return samples using exactly alpha empirical probability mass; distinguish it from upper-tail cost risk.

> **Hint**: sort, include $\lfloor \alpha N \rfloor$ full observations and the fractional boundary mass, then divide by $\alpha N$.

In [ ]:
def exercise_5(samples, alpha):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_5 before running learner checks")
    return ref.exercise_5(samples, alpha)


In [ ]:
assert abs(exercise_5(np.array([0,1,2,3,4.0]), 0.4) - 0.5) < 1e-9
assert np.isclose(exercise_5(np.array([0.,10.,20.]), .5), 10/3)
print('ok')

## Exercise 6 ★★ — Meaningful objective: train PPO-Lagrangian and audit

Train the agent on `LavaGridWorld` and verify discounted cost stays near budget. This is the *matching / safety meaningful objective* — in a real synthetic insurance or synthetic ATS deployment, the cost signal would be a fairness or capacity-saturation metric.

> **Hint**: call `ref.exercise_6(n_updates=20, cost_limit=1.5)`.

In [ ]:
def exercise_6(seed=0, n_updates=20, cost_limit=1.5):
    # YOUR CODE HERE: train and return a dict {history, final_cost, final_return}.
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_6 before running learner checks")
    return ref.exercise_6(seed=seed, n_updates=n_updates, cost_limit=cost_limit)


In [ ]:
out = exercise_6(seed=0, n_updates=20, cost_limit=1.5)
h = out['history']
fig, axes = plt.subplots(1,2, figsize=(10,3.5))
axes[0].plot(h['return'], color=RL_PALETTE['blue']); axes[0].set_title('Return')
axes[1].plot(h['cost'], color=RL_PALETTE['red']); axes[1].axhline(1.5, color=RL_PALETTE['gray'], linestyle=':')
axes[1].set_title('Cost vs budget'); plt.tight_layout(); plt.show()
print('final cost', out['final_cost'], 'final return', out['final_return'])

## Exercise 7 ★★★ — Open-ended: safe vs unsafe

Train with $d = 0.5$ vs $d = 10^6$. Plot return and cost for both. Explain (markdown) why the safe policy may have lower return — and why this is *acceptable*.

> **Hint**: `ref.exercise_7` returns the two histories.

In [ ]:
def exercise_7(seed=0, n_updates=20):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_7 before running learner checks")
    return ref.exercise_7(seed=seed, n_updates=n_updates)


In [ ]:
res = exercise_7(seed=0, n_updates=20)
fig, axes = plt.subplots(1,2, figsize=(10,3.5))
axes[0].plot(res['unsafe']['return'], label='unconstrained', color=RL_PALETTE['gray'])
axes[0].plot(res['safe']['return'], label='d=0.5', color=RL_PALETTE['blue']); axes[0].legend(); axes[0].set_title('Return')
axes[1].plot(res['unsafe']['cost'], label='unconstrained', color=RL_PALETTE['gray'])
axes[1].plot(res['safe']['cost'], label='d=0.5', color=RL_PALETTE['red']); axes[1].legend(); axes[1].set_title('Cost')
plt.tight_layout(); plt.show()

## Exercise 8 ★★★ — PID-Lagrangian one-step

Implement Stooke et al. (2020) PID-Lagrangian on $\lambda$.

> **Hint**: $\lambda \leftarrow \max(0, K_p e + K_i \sum_t e_t + K_d (J_c^{(t)} - J_c^{(t-1)}))$.

In [ ]:
def exercise_8(lam_prev, integral_prev, J_c_hist, d, Kp=0.1, Ki=0.05, Kd=0.05):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_8 before running learner checks")
    return ref.exercise_8(lam_prev, integral_prev, J_c_hist, d, Kp, Ki, Kd)


In [ ]:
lam, integ = exercise_8(0.0, 0.0, [2.0, 2.5], 1.0)
assert lam > 0.0 and integ > 0.0
print('lam', lam, 'integ', integ)
assert np.isclose(integ,1.5)
assert np.isclose(lam,.1*1.5+.05*1.5+.05*.5)


<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 ★ — Lagrangian dual step</summary>

> **Hint** (try yourself first): descent on this Lagrangian gives $\lambda \leftarrow [\lambda + \eta(J_c - d)]_+$. Sign matters.

</details>

<details>
<summary>Exercise 2 ★ — P3O hinge subgradient</summary>

> **Hint**: it is $\kappa$ above $d$, $0$ below $d$, anything in $[0,\kappa]$ at the kink.

</details>

<details>
<summary>Exercise 3 ★★ — GAE on the cost stream</summary>

> **Hint**: iterate from the end; $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ and $A_t = \delta_t + \gamma\lambda A_{t+1}$ when nonterminal.

</details>

<details>
<summary>Exercise 4 ★★ — FOCOPS closed-form non-parametric update</summary>

> **Hint**: log-space + softmax for numerical stability.

</details>

<details>
<summary>Exercise 5 ★★ — Empirical CVaR</summary>

> **Hint**: sort, include $\lfloor \alpha N \rfloor$ full observations and the fractional boundary mass, then divide by $\alpha N$.

</details>

<details>
<summary>Exercise 6 ★★ — Meaningful objective: train PPO-Lagrangian and audit</summary>

> **Hint**: call `ref.exercise_6(n_updates=20, cost_limit=1.5)`.

</details>

<details>
<summary>Exercise 7 ★★★ — Open-ended: safe vs unsafe</summary>

> **Hint**: `ref.exercise_7` returns the two histories.

</details>

<details>
<summary>Exercise 8 ★★★ — PID-Lagrangian one-step</summary>

> **Hint**: $\lambda \leftarrow \max(0, K_p e + K_i \sum_t e_t + K_d (J_c^{(t)} - J_c^{(t-1)}))$.

</details>

After solving, change one input or assumption and predict the result before
running it. In reference mode, passing checks verifies the demonstration. In learner mode,
the selected checks must pass against your completed functions without fallback.


## Statistical investigation — An expected cost budget does not determine tail risk

**Goal.** Distinguish expected cost, violation probability and a tail-cost criterion. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** Policy A incurs cost 4 with probability 0.1 and otherwise zero. Policy B
always incurs cost 0.4. Both have expected cost 0.4. Predict each probability
of exceeding cost 1 and each upper-tail cost CVaR at mass 0.1. Which constraint
does a CMDP's expected-cost budget actually specify?

**Work a small case by hand.** For the rare-cost policy E[C]=0.9×0+0.1×4=0.4, whereas P(C>1)=0.1
and upper-10% CVaR=4. For constant cost 0.4 the same mean accompanies
P(C>1)=0 and upper-10% CVaR=0.4. Separately, the empirical lower 50%
tail of (0,10,20) has mass 1.5 observations: include 0 plus half of 10,
then divide 5 by 1.5 to obtain 10/3, not the bottom-two average 5.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/09-statistical-claims.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
from common.statistics import empirical_cvar
costs={'Rare large cost':np.r_[np.zeros(90),np.full(10,4.)], 'Constant cost':np.full(100,.4)}
fig,axes=plt.subplots(1,2,figsize=(9,3.3))
for j,(name,c) in enumerate(costs.items()):
    print(name,{'expected_cost':c.mean(),'P(cost>1)':np.mean(c>1),
                'upper_CVaR_10pct':empirical_cvar(c,.1,'upper')})
    axes[0].hist(c,bins=np.linspace(0,4.4,12),weights=np.ones(len(c))/len(c),alpha=.5,label=name)
    axes[1].bar(j,empirical_cvar(c,.1,'upper'),label=name)
assert np.isclose(empirical_cvar([0,10,20],.5),10/3)
print('Fractional-boundary check [0,10,20], lower 50% CVaR =',empirical_cvar([0,10,20],.5))
axes[0].set(xlabel='Episode cost',ylabel='Probability mass');axes[0].legend(fontsize=8)
axes[1].set_xticks([0,1],['Rare large cost','Constant cost']);axes[1].set(ylabel='Upper-tail cost CVaR (10% mass)')
fig.tight_layout();plt.show()


**Interpret and transfer.** These are exact empirical distributions, not sampled safety guarantees.
Equal expected cost can coexist with very different violation probabilities
and tails. A learned multiplier changes training incentives; feasibility
requires evaluation against the stated cost definition.

**Transfer.** Interpret cost as a discounted stream rather than a single
episode event. Define the denominator and discount convention before using
a budget, then distinguish mean cost, chance constraints and tail constraints.


**Read your result.** Read expected cost, violation frequency and tail cost as different
quantities. The histogram uses specified probability masses; no claim of
population safety was estimated by training these two toy distributions.

<details><summary>Worked feedback for the changed assumption</summary>

A discounted cost sum C=sum gamma^t c_t needs an explicit horizon,
discount and population before its mean budget has units. A constraint on
E[C] does not impose C≤d for every episode. For nonnegative C, Markov's
inequality can only give the generally loose P(C≥b)≤E[C]/b; a chance or
CVaR constraint asks a different question.

</details>
